In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
import cv2


In [2]:
from nmf_utils import plot_contours_on_image

In [3]:
def main(path_save,file_stem):
    """
        Extract contours from final component masks and overlay them on the mean image.

        This function:
        1. Loads the mean image from the NMF preprocessing bundle.
        2. Loads the final (corrected) component masks.
        3. Extracts external contours for each component using OpenCV.
        4. Converts contour coordinates into a usable (x, y) format.
        5. Overlays contours on the mean image with predefined colors.
        6. Saves the visualization and updates the data bundle with contour information.

        Parameters
        ----------
        path_save : str
            Directory containing the input bundles and where outputs will be saved.
        file_stem : str
            Base filename used to locate input files and generate output filenames.

        Returns
        -------
        None
            Outputs are saved to disk and the input bundle is updated in-place.

        Outputs
        -------
        Saved files:
            - '{file_stem}_contours.png' : Image showing contours overlaid on mean image.

        Updated bundle:
            all_contours : list
                List of contours for each component.
                Structure:
                    [
                        [contour1 (Nx2 array), contour2, ...],   # component 0
                        [contour1, contour2, ...],              # component 1
                        ...
                    ]

        Notes
        -----
        - Only external contours are extracted (cv2.RETR_EXTERNAL).
        - Contours are stored as (N, 2) arrays representing (x, y) coordinates.
        - Colors are predefined and reused cyclically if components exceed available colors.
        - The mean image is scaled using percentile normalization (pmin=0.1, pmax=99.9).
        - Function overwrites the existing bundle with added contour data.
        """

    #Parameters   
    # Custom colors for plot_contours_on_image() function
    colors = np.array([
    [0.90, 0.10, 0.10],  # 0 red
    [0.10, 0.90, 0.80],  # 1 cyan
    [0.90, 0.40, 0.05],  # 2 deep orang
    [0.20, 0.10, 0.90],  # 3 blue-purple
    [0.05, 0.75, 0.25],  # 4 deeper green
    [0.80, 0.10, 0.90],  # 5 magenta
    [0.85, 0.80, 0.15],  # 6 yellow-orange
    [0.10, 0.30, 0.90],  # 7 blue
    [0.90, 0.10, 0.70],  # 8 pink
    [0.60, 0.90, 0.10],  # 9 light green
    [0.10, 0.50, 0.90],  # 10 light blue
    [0.60, 0.60, 0.60],  # 11 grey
    [0.10, 0.90, 0.55],  # 12 cyan‑gree
    [0.60, 0.10, 0.90],  # 13 purple
    [0.80, 0.90, 0.10],  # 14 yellow‑green
    [0.25, 0.40, 0.65],  # 15 steel blue
    [0.90, 0.30, 0.10],  # 16 orange‑red
    [0.50, 0.70, 0.50],  # 17 sage
    [0.10, 0.70, 0.90],  # 18 sky blue
    [0.75, 0.45, 0.10],  # 19 brown‑orange
    [0.40, 0.10, 0.90],  # 20 violet
    [0.25, 0.65, 0.45],  # 21 teal
    [0.90, 0.10, 0.50],  # 22 rose red
    [0.25, 0.55, 0.65],  # 23 blue‑teal
    [0.55, 0.25, 0.65],  # 24 plum
    [0.30, 0.90, 0.10],  # 25 bright green
    [0.70, 0.50, 0.70],  # 26 lavender
    [0.10, 0.60, 0.90],  # 27 mid blue
    [0.80, 0.45, 0.30],  # 28 peach
    [0.45, 0.65, 0.25],  # 29 muted green
    [0.65, 0.25, 0.50],  # 30 dusty rose
    [0.70, 0.60, 0.30],  # 31 khaki
    [0.50, 0.60, 0.70],  # 32 gray blue
    [0.65, 0.65, 0.10],  # 33 olive
    [0.80, 0.30, 0.40],  # 34 coral
    ])

    file_data_mean_image =os.path.join(path_save, file_stem +"_NMF_no_filt.pkl")
    with open(file_data_mean_image, "rb") as f:
        bundle_mean_image = pickle.load(f)
    mean_image=bundle_mean_image["mean_image"]    
    
    file_data =os.path.join(path_save, file_stem +"_final_components.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    final_components=bundle["final_components_corrected"]

    #Make contours
    all_contours = []
    for mask in final_components:
        contours, hierarchy = cv2.findContours(
            mask,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_NONE
        )

        xy_contours = []
        for contour in contours:
            # Remove the useless middle dimension: (N, 1, 2) -> (N, 2)
            xy = contour.squeeze(axis=1)
            xy_contours.append(xy)

        all_contours.append(xy_contours)

    # Plot contours and save the figure
    fig=plot_contours_on_image(mean_image, all_contours, colors,pmin=0.1, pmax=99.9)
    plt.savefig(os.path.join(path_save, file_stem+"_contours.png"), dpi=300)
    plt.close()

    # Add contours to the bundle and resave
    bundle["all_contours"] = all_contours
    # print(bundle.keys())
    with open(file_data, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    return
 

In [4]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [5]:
path_dist = "Data"
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        main(path_save,file_stem)
        print(f"Processing {file_stem} from {folder_exp}")


Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
